In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from rankci import (
    rank_confidence_intervals_bootstrap,
    rank_confidence_intervals_simulation,
    rank_ci_stepwise,
    rank_ci_stepwise_pairwise,
    compute_pairwise,
    load_spf,
    load_rtdsm,
    compute_errors,
    compute_squared_error_panel,
    select_top_forecasters,
    winsorize_panel,
)

# Data Loading

**Forecasts**: SPF microdata (individual-level NGDP forecasts by quarter and horizon)

**Realizations**: RTDSM advance estimates (`NOUTPUTQvQd.xlsx`) — the first-published GDP figure for each quarter, avoiding contamination from later revisions.

In [2]:
df = load_spf("../data/SPFmicrodata.xlsx")
print(f"SPF NGDP: {df.shape[0]} rows × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
df.head()

SPF NGDP: 9145 rows × 12 columns
Columns: ['YEAR', 'QUARTER', 'ID', 'INDUSTRY', 'NGDP1', 'NGDP2', 'NGDP3', 'NGDP4', 'NGDP5', 'NGDP6', 'NGDPA', 'NGDPB']


/Users/Parimah/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,YEAR,QUARTER,ID,INDUSTRY,NGDP1,NGDP2,NGDP3,NGDP4,NGDP5,NGDP6,NGDPA,NGDPB
0,1968,4,1,NaN,871.0,884.0,895.0,907.0,920.0,938.0,NaN,NaN
1,1968,4,2,NaN,871.0,891.0,910.0,929.0,958.0,973.0,NaN,NaN
2,1968,4,3,NaN,871.0,883.0,894.0,906.0,924.0,944.0,NaN,NaN
3,1968,4,4,NaN,871.0,885.0,891.0,902.0,919.0,937.0,NaN,NaN
4,1968,4,5,NaN,871.0,895.0,913.0,935.0,940.0,970.0,NaN,NaN


In [3]:
noutput = load_rtdsm("../data/NOUTPUTQvQd.xlsx")
print(f"RTDSM: {noutput.shape[0]} rows × {noutput.shape[1]} columns")
noutput.head()

RTDSM: 316 rows × 242 columns


NOUTPUT65Q4  NOUTPUT66Q1  NOUTPUT66Q2  NOUTPUT66Q3  NOUTPUT66Q4  \
YEAR QUARTER                                                                    
1947 1              223.6        223.6        223.6        223.6        223.6   
     2              227.6        227.6        227.6        227.6        227.6   
     3              231.8        231.8        231.8        231.8        231.8   
     4              242.1        242.1        242.1        242.1        242.1   
1948 1              248.0        248.0        248.0        248.0        248.0   

              NOUTPUT67Q1  NOUTPUT67Q2  NOUTPUT67Q3  NOUTPUT67Q4  NOUTPUT68Q1  \
YEAR QUARTER                                                                    
1947 1              223.6        223.6        223.6        223.6        223.6   
     2              227.6        227.6        227.6        227.6        227.6   
     3              231.8        231.8        231.8        231.8        231.8   
     4              242.1        242.1        242.1        242.1        242.1   
1948 1              248.0        248.0        248.0        248.0        248.0   

              ...  NOUTPUT23Q4  NOUTPUT24Q1  NOUTPUT24Q2  NOUTPUT24Q3  \
YEAR QUARTER  ...                                                       
1947 1        ...        243.2        243.2        243.2        243.2   
     2        ...        246.0        246.0        246.0        246.0   
     3        ...        249.6        249.6        249.6        249.6   
     4        ...        259.7        259.7        259.7        259.7   
1948 1        ...        265.7        265.7        265.7        265.7   

              NOUTPUT24Q4  NOUTPUT25Q1  NOUTPUT25Q2  NOUTPUT25Q3  NOUTPUT25Q4  \
YEAR QUARTER                                                                    
1947 1              243.2        243.2        243.2        243.2        243.2   
     2              246.0        246.0        246.0        246.0        246.0   
     3              249.6        249.6        249.6        249.6        249.6   
     4              259.7        259.7        259.7        259.7        259.7   
1948 1              265.7        265.7        265.7        265.7        265.7   

              NOUTPUT26Q1  
YEAR QUARTER               
1947 1              243.2  
     2              246.0  
     3              249.6  
     4              259.7  
1948 1              265.7  

[5 rows x 242 columns]

# Sanity Checks

In [4]:
# Verify advance estimates for a few known quarters
from rankci.data import advance_vintage_col, get_advance_estimate

test_cases = [(1995, 2), (1999, 2), (2008, 4), (2020, 2)]
print(f"{'Target Quarter':<20} {'Vintage Col':<18} {'Value (bn $)'}")
print("-" * 55)
for y, q in test_cases:
    col = advance_vintage_col(y, q)
    val = get_advance_estimate(y, q, noutput)
    print(f"  {y}:Q{q:<16}  {col:<18}  {val:,.1f}")

Target Quarter       Vintage Col        Value (bn $)
-------------------------------------------------------
  1995:Q2                 NOUTPUT95Q3         7,011.8
  1999:Q2                 NOUTPUT99Q3         8,893.3
  2008:Q4                 NOUTPUT09Q1         14,264.6
  2020:Q2                 NOUTPUT20Q3         19,408.8


# Forecast Errors

In [5]:
# Compute all forecast errors (all horizons)
errors_df = compute_errors(df, noutput)

# Summary by horizon
print("Error summary by horizon:")
for h in range(1, 7):
    col = f"error_NGDP{h}"
    if col in errors_df.columns:
        print(f"  {col}: mean={errors_df[col].mean():+.1f}, "
              f"std={errors_df[col].std():.1f}, "
              f"nan%={errors_df[col].isna().mean()*100:.1f}%")

errors_df.head()

Error summary by horizon:
  error_NGDP1: mean=+0.3, std=6.1, nan%=6.1%
  error_NGDP2: mean=-15.0, std=118.9, nan%=6.5%
  error_NGDP3: mean=-20.4, std=258.5, nan%=6.9%
  error_NGDP4: mean=-25.4, std=313.2, nan%=7.5%
  error_NGDP5: mean=-30.7, std=372.2, nan%=7.9%
  error_NGDP6: mean=-32.2, std=441.6, nan%=12.7%


,YEAR,QUARTER,ID,INDUSTRY,error_NGDP1,error_NGDP2,error_NGDP3,error_NGDP4,error_NGDP5,error_NGDP6
0,1968,4,1,NaN,0.2,-3.8,-8.4,-18.1,-22.3,-15.1
1,1968,4,2,NaN,0.2,3.2,6.6,3.9,15.7,19.9
2,1968,4,3,NaN,0.2,-4.8,-9.4,-19.1,-18.3,-9.1
3,1968,4,4,NaN,0.2,-2.8,-12.4,-23.1,-23.3,-16.1
4,1968,4,5,NaN,0.2,7.2,9.6,9.9,-2.3,16.9


In [6]:
# Squared error panel for NGDP3 (one-quarter-ahead)
# Rows = (YEAR, QUARTER), Columns = forecaster IDs, Values = squared errors
X_wide = compute_squared_error_panel(df, noutput, horizon="NGDP3")
print(f"Panel shape: {X_wide.shape[0]} quarters × {X_wide.shape[1]} forecasters")
X_wide.head()

Panel shape: 225 quarters × 345 forecasters


ID               1      2       3       4      5       6      7       8    \
YEAR QUARTER                                                                
1968 4         70.56  43.56   88.36  153.76  92.16   70.56  70.56  237.16   
1969 1           NaN   1.21  292.41  445.21    NaN  146.41  16.81  292.41   
     2        234.09    NaN  151.29  176.89    NaN  299.29    NaN  151.29   
     3         37.21    NaN   37.21     NaN    NaN   37.21    NaN     NaN   
     4         43.56    NaN    2.56   92.16    NaN   43.56  31.36   31.36   

ID               9       10   ...  596  599  600  601  602  603  604  605  \
YEAR QUARTER                  ...                                           
1968 4        108.16   70.56  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
1969 1        171.61  123.21  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
     2        334.89   39.69  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
     3           NaN    8.41  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
     4         43.56     NaN  ...  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   

ID            606  607  
YEAR QUARTER            
1968 4        NaN  NaN  
1969 1        NaN  NaN  
     2        NaN  NaN  
     3        NaN  NaN  
     4        NaN  NaN  

[5 rows x 345 columns]

# Forecaster Selection

In [7]:
# Observation counts and mean MSE per forecaster
obs_counts = X_wide.notna().sum()
mean_mse   = X_wide.mean()

stats = pd.DataFrame({
    "obs_count": obs_counts,
    "mean_mse":  mean_mse,
    "rmse":      np.sqrt(mean_mse),
}).sort_values("obs_count", ascending=False)

print(f"Total forecasters: {len(stats)}")
print(f"Forecasters with >= 20 obs: {(stats['obs_count'] >= 20).sum()}")
print(f"Forecasters with >= 50 obs: {(stats['obs_count'] >= 50).sum()}")
stats.head(20)

Total forecasters: 345
Forecasters with >= 20 obs: 148
Forecasters with >= 50 obs: 52


,obs_count,mean_mse,rmse
ID,,,
65,122,8634.247135,92.920650
426,121,122073.450497,349.390112
433,119,157364.258540,396.691642
84,119,9874.065367,99.368332
428,117,105996.918432,325.571679
411,115,133744.990174,365.711622
421,114,106525.187449,326.381966
40,107,7483.908598,86.509587
510,103,151490.430716,389.217716


In [8]:
# Select top-N forecasters (adjust N here)
N = 8
X_panel = select_top_forecasters(X_wide, N=N, min_obs=20)
print(f"Selected {X_panel.shape[1]} forecasters, {X_panel.shape[0]} quarters")
print(f"Forecaster IDs: {X_panel.columns.tolist()}")

Selected 8 forecasters, 225 quarters
Forecaster IDs: [65, 426, 84, 433, 428, 411, 421, 40]


# Rank Confidence Intervals

Using the stepwise bootstrap with pairwise NW-HAC standard errors (handles the unbalanced panel).

In [9]:
X = X_panel.values
population_ids = X_panel.columns.tolist()

out = rank_ci_stepwise_pairwise(X, alpha=0.05, B=5000, seed=42)

results = pd.DataFrame({
    "ID":         population_ids,
    "MSE":        out["theta_hat"].round(1),
    "RMSE":       np.sqrt(out["theta_hat"]).round(1),
    "CI_lower":   out["rank_ci"][:, 0],
    "CI_upper":   out["rank_ci"][:, 1],
}).sort_values("MSE")
results.index = range(1, len(results) + 1)
results.index.name = "Rank"
results

=== Pairwise shared observations ===
  Min: 33, Mean: 72.1, Max: 106
  Pairs with < 20 shared obs: 0

=== Test statistics (delta_hat / se) ===
  Max: 2.8941, Pairs with t > 1.96: 2


,ID,MSE,RMSE,CI_lower,CI_upper
Rank,,,,,
1,40,7483.9,86.5,1,8
2,65,8634.2,92.9,1,8
3,84,9874.1,99.4,1,8
4,428,105996.9,325.6,1,8
5,421,106525.2,326.4,1,8
6,426,122073.5,349.4,1,8
7,411,133745.0,365.7,1,8
8,433,157364.3,396.7,1,8


# Worst-Quarter Inspection

The CIs are uninformative — every forecaster gets rank `[1, 8]`. To understand why, look at which quarters dominate the squared-error series and whether they reflect genuine forecast misses or data artifacts.

In [ ]:
# Average squared error per quarter, across the selected forecasters
period_mse = X_panel.mean(axis=1)

print(f"Time coverage: {X_panel.index[0]} to {X_panel.index[-1]} ({len(X_panel)} quarters)\n")
print("Top 10 quarters with highest average squared error:")
print(period_mse.nlargest(10))

# Plot MSE over time
n = len(X_panel)
plt.figure(figsize=(12, 4))
plt.plot(range(n), period_mse, marker="o", linewidth=1, markersize=3)
plt.xticks(
    ticks=range(0, n, 8),
    labels=[f"{y}Q{q}" for y, q in X_panel.index[::8]],
    rotation=45,
)
plt.ylabel("Average squared error (bn² $)")
plt.title(f"Avg squared NGDP3 forecast error across {N} forecasters")
plt.tight_layout()
plt.show()

In [ ]:
# Recover the original SPF level forecasts for the worst quarters
# to confirm the errors reflect genuine forecast misses (not data alignment bugs).
worst_quarters = period_mse.nlargest(3).index.tolist()

print("Original SPF NGDP3 forecasts vs advance estimate, worst quarters:\n")
for (y, q) in worst_quarters:
    actual = get_advance_estimate(y, q, noutput)

    # NGDP3 targeting (y, q) was submitted in survey (y, q-1)
    survey_y, survey_q = (y, q - 1) if q > 1 else (y - 1, 4)

    subset = df[
        (df["YEAR"] == survey_y) &
        (df["QUARTER"] == survey_q) &
        (df["ID"].isin(population_ids))
    ][["ID", "NGDP3"]]

    print(f"Target {y}Q{q}  (survey {survey_y}Q{survey_q})  actual = {actual:,.1f}")
    for _, row in subset.iterrows():
        err = row["NGDP3"] - actual
        print(f"    ID {int(row['ID']):>4}  forecast = {row['NGDP3']:>10,.1f}   error = {err:>+10,.1f}")
    print()

# Winsorization

Crisis quarters (COVID, dot-com peak) inflate the variance of the pairwise difference series, which widens the SEs and the bootstrap critical value. Two ways to limit their influence:

1. **Winsorize the squared-error panel** before computing pairwise differences (clip top tail per forecaster).
2. **Winsorize each pairwise difference series** symmetrically, just before the NW SE.

Both are exploratory — we want to see if the gap between max test statistic and bootstrap critical value closes.

In [ ]:
# Variant 1: winsorize the squared-error panel per-forecaster (upper tail only)
print("=== Variant 1: winsorize the panel ===")
print(f"{'upper_pct':<12} {'max_t':<10} {'mean CI width':<16}")
print("-" * 40)
for upper_pct in [99, 97, 95, 90]:
    Xw = winsorize_panel(X, upper_pct=upper_pct)
    delta_w, se_w, _ = compute_pairwise(Xw, se_method="nw")
    t_vals = (delta_w / se_w)[~np.isnan(se_w)]

    out_w = rank_ci_stepwise_pairwise(Xw, alpha=0.05, B=1000, seed=42, verbose=False)
    widths = out_w["rank_ci"][:, 1] - out_w["rank_ci"][:, 0]

    print(f"  {upper_pct:<10} {t_vals.max():<10.3f} {widths.mean():<16.2f}")

In [ ]:
# Variant 2: winsorize each pairwise difference series (symmetric, in the SE itself)
print("=== Variant 2: winsorize differences ===")
print(f"{'winsor_pct':<12} {'max_t':<10} {'mean CI width':<16}")
print("-" * 40)
for winsor_pct in [99, 97, 95, 90]:
    delta_w, se_w, _ = compute_pairwise(X, se_method="nw", winsor_pct=winsor_pct)
    t_vals = (delta_w / se_w)[~np.isnan(se_w)]

    out_w = rank_ci_stepwise_pairwise(
        X, alpha=0.05, B=1000, seed=42, winsor_pct=winsor_pct, verbose=False,
    )
    widths = out_w["rank_ci"][:, 1] - out_w["rank_ci"][:, 0]

    print(f"  {winsor_pct:<10} {t_vals.max():<10.3f} {widths.mean():<16.2f}")